### This notebook compute "VSS6. Volume of annual water demand for population use" indicator for the 27 basins of IKI Project

Spanish: Volumen de la demanda anual del agua para uso poblacional

**Created:** 12/2025 by Sophia Bakar (sbakar@rti.org)

**Project #:** 0219481  

**Last modified:** 12/22/2025

**Status:** Complete (for baseline scenario)

**QA Status:** reviewed by Scott Sheeder  

**Original Script Stored at:** Research Triangle Institute\IKI Peru Project - General\Interno\AI2a_Metodologia\Indicadores\Vulnerabilidad
 
**Inputs:**  waterALLOC output database

**Outputs:** 
 
**Assumptions:** 
 
**Future work:** 
 
**Notes:** Here we directly query the waterALLOC output SQL database and import the results to the indicators database. We have included some code to check the min and max values of the results and the min and max values listed in the indicators database to ensure they align. Code is provided to update the min and max values in the indicators DB as needed. 

In [1]:
#import numpy as np
import pandas as pd
import sqlite3
import geopandas as gpd
from scipy.spatial import cKDTree
import matplotlib.pyplot as plt
from tqdm import tqdm
import itertools

In [2]:
# set up user and database path
#user = 'jmayo'
#user= 'cpickering'
#user = 'sgilson'
#user = 'nreynolds'
user = 'sbakar'
#db_path = fr'C:\Users\{user}\Research Triangle Institute\IKI Peru Project - General\Interno\AI2a_Metodologia\Indicadores\BD_RiesgoClimatico_IKI.db'
db_path = fr"C:\Users\sbakar\OneDrive - Research Triangle Institute\IKI Peru Project - General\Interno\AI2a_Metodologia\Indicadores\BD_RiesgoClimatico_IKI.db"
# wateralloc_db = fr"C:\Users\{user}\Research Triangle Institute\IKI Peru Project - General\Interno\AI2b_Modelacion\Grupos_Modelacion\Resultados\BalanceHidrico.sqlite"
wateralloc_db = fr"C:\Users\sbakar\OneDrive - Research Triangle Institute\IKI Peru Project - General\Interno\AI2b_Modelacion\Grupos_Modelacion\Resultados\BalanceHidrico.sqlite"

In [4]:
# set up indicator ID and get scenarios from database
IndID = 406 #Indicator ID (Exposure = 2 + 0X where X is the Exposure Indicator number, Peligro= 1 +0x, VSB= 3 +0x, VSS= 4 +0x, VCA= 5 +0x)
# Connect to Indicators DB and get available scenarios
conn = sqlite3.connect(db_path)

wa_scenarios_df = pd.read_sql_query(
    """
    SELECT WaScnID, WaScnName
    FROM WaScenarios
    ORDER BY WaScnID
    """,
    conn
)

conn.close()

# For now: only baseline and first future
scenario_ids = wa_scenarios_df.loc[
    wa_scenarios_df['WaScnID'].isin([1, 3]), 'WaScnID'
].tolist()

# for all scenarios:
#scenario_ids = wa_scenarios_df['WaScnID'].tolist()

In [5]:
#subbasins_shapefile = f'C:/Users/{user}/Research Triangle Institute/IKI Peru Project - General/Interno/AI2b_Modelacion/Grupos_Modelacion/GIS_WaterALLOC_General/Peru_AHD_with_districts.shp'
subbasins_shapefile = fr"C:\Users\sbakar\OneDrive - Research Triangle Institute\IKI Peru Project - General\Interno\AI2b_Modelacion\Grupos_Modelacion\GIS_WaterALLOC_General\Peru_AHD_with_districts.shp"
subbasins_gdf = gpd.read_file(subbasins_shapefile).set_index('COMID').to_crs('WGS84')

In [6]:
dfs = []

conn_wa = sqlite3.connect(wateralloc_db)

query_demanda = """
SELECT 
    a.comid AS COMID,
    a.[Demanda] AS Demanda,
    a.[Suministro] AS Suministro
FROM "WAMSS_Demanda anual promedio por tipo de demanda por COMID" AS a
JOIN WAMMS_RunsInfo AS b 
    ON a.RunID = b.RunID
JOIN Scenarios AS c 
    ON c.WaScnID = b.WaScnID
WHERE c.WaScnID = ?
  AND a.Sector = ?
"""

sector_name = "Poblacional"

for scn_id in scenario_ids:
    df = pd.read_sql_query(
        query_demanda,
        conn_wa,
        params=(scn_id, sector_name)
    )

    df["WaScnID"] = scn_id

    dfs.append(df)

# Combine all scenarios
demanda_df = pd.concat(dfs, ignore_index=True)

# Close connection
conn_wa.close()

In [7]:
# Create full COMID x scenario combinations (Cartesian product)
all_comids = subbasins_gdf.reset_index()[['COMID']]
all_scenarios = pd.DataFrame({'WaScnID': scenario_ids})

all_comid_scenarios = pd.DataFrame(
    list(itertools.product(all_comids['COMID'], all_scenarios['WaScnID'])),
    columns=['COMID', 'WaScnID']
)

# Merge demanda data
demanda_gdf = all_comid_scenarios.merge(
    demanda_df,
    on=['COMID', 'WaScnID'],
    how='left'
)

# Now, **fill missing values for Demanda and Suministro** with 0
demanda_gdf['Demanda'] = demanda_gdf['Demanda'].fillna(0)
demanda_gdf['Suministro'] = demanda_gdf['Suministro'].fillna(0)

# **WaScnID is already populated from all_comid_scenarios**, so no NaNs here

# Merge geometry
subbasins_reset = subbasins_gdf.reset_index()[['COMID', 'geometry']]
demanda_gdf = demanda_gdf.merge(subbasins_reset, on='COMID', how='left')

# Quick check
print(f"Total rows (COMID x scenario): {len(demanda_gdf)}")
print(f"Rows with Demanda >0: {(demanda_gdf['Demanda'] > 0).sum()}")
print(f"Rows with Demanda =0: {(demanda_gdf['Demanda'] == 0).sum()}")
print(f"Any missing WaScnID? {demanda_gdf['WaScnID'].isna().sum()}")


Total rows (COMID x scenario): 7310
Rows with Demanda >0: 112
Rows with Demanda =0: 7198
Any missing WaScnID? 0


In [8]:
demanda_gdf

,COMID,WaScnID,Demanda,Suministro,geometry
0,311153600,1,0.0,0.0,"POLYGON ((-69.91667 -16.0375, -69.91667 -16.01..."
1,311153600,3,0.0,0.0,"POLYGON ((-69.91667 -16.0375, -69.91667 -16.01..."
2,310832400,1,0.0,0.0,"POLYGON ((-70.0125 -15.27917, -70.00833 -15.27..."
3,310832400,3,0.0,0.0,"POLYGON ((-70.0125 -15.27917, -70.00833 -15.27..."
4,310825700,1,0.0,0.0,"POLYGON ((-70.15 -15.31667, -70.14583 -15.3166..."
...,...,...,...,...,...
7305,312066700,3,0.0,0.0,"MULTIPOLYGON (((-70.2625 -18.025, -70.2625 -18..."
7306,320251000,1,0.0,0.0,"POLYGON ((-70.42917 -18.2625, -70.42917 -18.27..."
7307,320251000,3,0.0,0.0,"POLYGON ((-70.42917 -18.2625, -70.42917 -18.27..."
7308,320254600,1,0.0,0.0,"POLYGON ((-76.925 -12.02917, -76.925 -12.03333..."


In [9]:
conn = sqlite3.connect(db_path)
cursor = conn.cursor()
delete_query = """
DELETE FROM IndValues_WaALLOC
WHERE IndID = ?;
"""
cursor.execute(delete_query, (IndID,))
conn.commit()
print(f"Deleted existing rows for IndID = {IndID}")
conn.close()

Deleted existing rows for IndID = 406


In [10]:
# insert data into the sqlite database
# Connect to SQLite database
conn = sqlite3.connect(db_path)
cursor = conn.cursor()

rows_to_insert = []
for _, row in demanda_gdf.iterrows():
    rows_to_insert.append((row['WaScnID'], IndID, row['COMID'], row['Demanda']))


# Insert data into IndValues_Dyn
insert_query = """
INSERT OR REPLACE INTO IndValues_WaALLOC (WaScnID, IndID, COMID, Value)
VALUES (?, ?, ?, ?);
"""

In [11]:
# Check that min and max values match the expected range based on the Indicators Table

indicator_limits = pd.read_sql_query(
    """
    SELECT IndID, Min, Max
    FROM Indicators
    WHERE IndID = ?
    """,
    conn,
    params=(IndID,)
)

if indicator_limits.empty:
    raise ValueError(f"No entry found in Indicators table for IndID = {IndID}")

ind_min = indicator_limits.loc[0, 'Min']
ind_max = indicator_limits.loc[0, 'Max']

print(f"\nIndicator {IndID} limits from Indicators table -> Min: {ind_min}, Max: {ind_max}")

# Compute value stats by scenario
value_stats = demanda_gdf.groupby('WaScnID')['Demanda'].agg(['min', 'max', 'count']).reset_index()
print("\n=== Values to be inserted (by scenario) ===")
print(value_stats)

# Check for duplicates in the rows to insert
df_check = pd.DataFrame(rows_to_insert, columns=['WaScnID', 'IndID', 'COMID', 'Value'])
duplicates = df_check.duplicated(subset=['WaScnID', 'IndID', 'COMID'])
print("\nDuplicates in rows_to_insert:")
print(df_check[duplicates])


Indicator 406 limits from Indicators table -> Min: 0.9499999999999998, Max: 45699.46500000003

=== Values to be inserted (by scenario) ===
   WaScnID  min      max  count
0        1  0.0  43523.3   3655
1        3  0.0  43523.3   3655

Duplicates in rows_to_insert:
Empty DataFrame
Columns: [WaScnID, IndID, COMID, Value]
Index: []


In [12]:
#Execute insert to SQLite Database
cursor.executemany(insert_query, rows_to_insert)
conn.commit()
conn.close()